verification.py -- independent checks of the skip-entry code

Convergence alone shows the code is *consistent*; it does not show the equations are *right*
(a wrong equation converges just as cleanly). So every check here compares against something we
know independently of model.py / euler.py:

  1. euler.py   vs closed-form results for the *discrete* Euler map (and the continuous solution)
  2. model.py   vs a second, from-scratch implementation of the equations of motion
  3. model.py + euler.py in the vacuum limit: energy and angular momentum must be conserved
  4. Allen-Eggers limit (no gravity, no lift, straight path): exact solution of a special case
  5. Whole pipeline vs an independent RK4 integration of the independently written equations

Run:   python verification.py        (about 30 s; exit code 1 if any check fails)
Used by convergence.py, which imports the interface block and reference tools below.

In [1]:
import sys
import numpy as np

# =====================================================================================
# Assumed: SI units, state = [h (m), V (m/s), gamma (rad, + = climbing), theta (rad)].
# =====================================================================================
import parameters as P      # constants, initial conditions, thresholds
import model as M           # density, forces, load, state derivatives
import euler as E           # forward Euler time marching

MU, R_E, G0 = P.MU, P.R_EARTH, P.G0
RHO0, H_SCALE = P.RHO0, P.H_SCALE
MASS, AREA, CD, CL = P.MASS, P.AREA, P.CD, P.CL          # CL = (L/D) * CD
H0, V0, GAMMA0 = P.H0, P.V0, P.GAMMA0                    # baseline entry (gamma0 in radians)
V_CAPTURE = getattr(P, "V_CAPTURE", 1000.0)              # [m/s] below this the capsule cannot climb back
T_MAX = getattr(P, "T_MAX", 1500.0)                      # [s] hard stop


def team_rhs(state):
    """Four state derivatives from model.py -> ndarray (4,)."""
    return np.asarray(M.state_derivatives(state), float)


def team_load(state):
    """Aerodynamic load factor in g from model.py -> float."""
    return float(M.load_factor(state))


def team_euler(rhs, state0, dt, n_steps):
    """Forward-Euler marching from euler.py -> array (n_steps+1, 4), row 0 = initial state."""
    return np.asarray(E.forward_euler(rhs, state0, dt, n_steps), float)
# =====================================================================================


# ---------------------------------------------------------------- independent reference tools
def ref_rhs(state, g_on=1.0, cl=CL):
    """Second implementation of the equations of motion, written straight from the report.
    g_on = 0 switches off gravity AND the curvature term (flat-Earth, no-gravity limit)."""
    h, V, gam, _ = state
    r = R_E + h
    g = MU / r**2
    q = 0.5 * RHO0 * np.exp(-h / H_SCALE) * V * V
    D, L = q * CD * AREA, q * cl * AREA
    return np.array([V * np.sin(gam),
                     -D / MASS - g_on * g * np.sin(gam),
                     L / (MASS * V) + g_on * (V / r - g / V) * np.cos(gam),
                     V * np.cos(gam) / r])


def ref_load(state):
    h, V = state[0], state[1]
    q = 0.5 * RHO0 * np.exp(-h / H_SCALE) * V * V
    return q * AREA * np.hypot(CD, CL) / (MASS * G0)


def euler_step(state, dt):
    """One forward-Euler step of the team model. Check 1 shows this equals euler.py's marching."""
    return state + dt * team_rhs(state)


def rk4_step(f, s, dt):
    k1 = f(s); k2 = f(s + 0.5 * dt * k1); k3 = f(s + 0.5 * dt * k2); k4 = f(s + dt * k3)
    return s + dt * (k1 + 2 * k2 + 2 * k3 + k4) / 6.0


def march(step, s0, T, dt):
    """Fixed-step march to t = T (T must be a multiple of dt); returns the full history (n+1, 4)."""
    n = int(round(T / dt))
    assert abs(n * dt - T) < 1e-9 * max(1.0, T), f"T={T} is not a multiple of dt={dt}"
    hist = np.empty((n + 1, len(s0)))
    hist[0] = s0
    for k in range(n):
        hist[k + 1] = step(hist[k], dt)
    return hist


def specific_energy(s):
    return 0.5 * s[..., 1] ** 2 - MU / (R_E + s[..., 0])


def ang_momentum(s):
    return (R_E + s[..., 0]) * s[..., 1] * np.cos(s[..., 2])


def tail_slope(dts, errs, k=4):
    """Observed order from the k finest step sizes (the asymptotic regime)."""
    dts, errs = np.asarray(dts, float), np.asarray(errs, float)
    idx = np.argsort(dts)[:k]
    return np.polyfit(np.log(dts[idx]), np.log(errs[idx]), 1)[0]


DTS = np.array([0.4, 0.2, 0.1, 0.05, 0.025, 0.0125])
S0 = np.array([H0, V0, GAMMA0, 0.0])
RESULTS = []


def report(name, passed, detail):
    RESULTS.append((name, passed))
    print(f"  [{'PASS' if passed else 'FAIL'}] {name}: {detail}")


def table(header, rows):
    print("    " + "  ".join(f"{h:>15}" for h in header))
    for r in rows:
        print("    " + "  ".join(f"{x:15.6g}" for x in r))


# ---------------------------------------------------------------- 1. euler.py
def check_euler():
    print("\n1. euler.py against closed-form results")
    s0 = np.array([1.0, 2.0, 3.0, 4.0])
    # (a) y' = -lam*y. Forward Euler gives exactly y_n = (1 - lam*dt)^n y_0.
    lam, dt, n = 2.0, 0.05, 40
    got = team_euler(lambda s: -lam * s, s0, dt, n)[-1]
    err = np.max(np.abs(got - s0 * (1 - lam * dt) ** n) / np.abs(s0))
    report("exact discrete map, decay", err < 1e-12, f"max rel. diff = {err:.1e}")
    # (b) x'' = -w^2 x. Forward Euler multiplies the amplitude^2 by exactly (1 + (w dt)^2) per step.
    w, dt, n = 3.0, 0.02, 100
    osc = lambda s: np.array([s[1], -w * w * s[0], 0.0, 0.0])
    s = team_euler(osc, np.array([1.0, 0.0, 0.0, 0.0]), dt, n)[-1]
    amp2 = s[0] ** 2 + (s[1] / w) ** 2
    err = abs(amp2 / (1 + (w * dt) ** 2) ** n - 1)
    report("exact discrete map, oscillator", err < 1e-10, f"amplitude growth matches (1+(w dt)^2)^n, rel. diff = {err:.1e}")
    # (c) order of accuracy against the true solution of y' = -lam*y
    T = 2.0
    errs = [abs(team_euler(lambda s: -lam * s, s0, d, int(round(T / d)))[-1][0] - np.exp(-lam * T)) for d in DTS]
    p = tail_slope(DTS, errs)
    report("order on y' = -lam*y", 0.9 < p < 1.1, f"observed slope = {p:.3f} (expect 1)")
    # (d) euler.py on the real model equals the plain recursion s + dt*f(s)
    dt, n = 0.05, 2000
    a = team_euler(team_rhs, S0, dt, n)[-1]
    b = S0.copy()
    for _ in range(n):
        b = euler_step(b, dt)
    err = np.max(np.abs(a - b) / np.maximum(np.abs(b), 1e-12))
    report("euler.py equals s + dt*f(s) on model.py", err < 1e-9, f"max rel. diff after {n} steps = {err:.1e}")


# ---------------------------------------------------------------- 2. model.py vs second implementation
def check_model():
    print("\n2. model.py against an independent implementation of the equations")
    rng = np.random.default_rng(0)
    states = [S0] + [np.array([rng.uniform(0, 122e3), rng.uniform(500, 11500),
                               np.deg2rad(rng.uniform(-20, 20)), 0.0]) for _ in range(300)]
    worst, worst_load = 0.0, 0.0
    for s in states:
        a, b = team_rhs(s), ref_rhs(s)
        worst = max(worst, np.max(np.abs(a - b) / np.maximum(np.abs(b), 1e-10)))
        worst_load = max(worst_load, abs(team_load(s) - ref_load(s)) / max(ref_load(s), 1e-300))
    report("state derivatives", worst < 1e-9, f"max rel. diff over {len(states)} states = {worst:.1e}")
    report("load factor (g)", worst_load < 1e-9, f"max rel. diff = {worst_load:.1e}")


# ---------------------------------------------------------------- 3. vacuum conservation
def check_vacuum():
    print("\n3. Vacuum limit (start at 600 km where density is ~1e-36 kg/m^3): energy and angular momentum conserved")
    s0 = np.array([600e3, V0, GAMMA0, 0.0])
    e0, l0 = specific_energy(s0), ang_momentum(s0)
    rows = []
    for dt in DTS:
        s = team_euler(team_rhs, s0, dt, int(round(300.0 / dt)))[-1]
        rows.append((dt, abs(specific_energy(s) - e0) / abs(e0), abs(ang_momentum(s) - l0) / l0))
    rows = np.array(rows)
    table(["dt [s]", "rel energy err", "rel h-mom err"], rows)
    pe, pl = tail_slope(rows[:, 0], rows[:, 1]), tail_slope(rows[:, 0], rows[:, 2])
    report("conserved quantities", 0.9 < pe < 1.1 and 0.9 < pl < 1.1 and rows[-1, 1] < 1e-5,
           f"slopes: energy {pe:.2f}, angular momentum {pl:.2f} (expect 1); error -> 0 as dt -> 0")
    return rows


# ---------------------------------------------------------------- 4. Allen-Eggers
def check_allen_eggers():
    print("\n4. Allen-Eggers limit: no gravity, no lift, straight path")
    gam0, T = np.deg2rad(-10.0), 80.0
    s0 = np.array([H0, V0, gam0, 0.0])
    f = lambda s: ref_rhs(s, g_on=0.0, cl=0.0)
    beta = MASS / (CD * AREA)
    rho = lambda h: RHO0 * np.exp(-h / H_SCALE)
    a_exact = -V0**2 * np.sin(gam0) / (2 * np.e * H_SCALE)          # peak deceleration
    rows = []
    for dt in DTS:
        S = march(lambda s, d: s + d * f(s), s0, T, dt)
        decel = 0.5 * rho(S[:, 0]) * S[:, 1] ** 2 / beta
        V_ex = V0 * np.exp(H_SCALE * (rho(S[:, 0]) - rho(H0)) / (2 * beta * np.sin(gam0)))   # V(h)
        rows.append((dt, abs(decel.max() - a_exact) / a_exact, np.max(np.abs(S[:, 1] - V_ex)) / V0))
    rows = np.array(rows)
    table(["dt [s]", "peak-decel err", "max|V-V_AE|/V0"], rows)
    p = tail_slope(rows[:, 0], rows[:, 2])
    report("Allen-Eggers V(h) and peak deceleration", 0.9 < p < 1.15 and rows[-1, 1] < 0.01,
           f"V(h) slope {p:.2f} (expect 1); peak decel within {rows[-1,1]*100:.2f}% at dt={DTS[-1]} s")
    return rows


# ---------------------------------------------------------------- 5. whole pipeline vs independent RK4
def check_pipeline(T=100.0):
    print(f"\n5. model.py + euler.py vs independent RK4 on independently coded equations (t = {T:.0f} s, baseline entry)")
    ref = march(lambda s, d: rk4_step(ref_rhs, s, d), S0, T, 0.05)[-1]
    ref2 = march(lambda s, d: rk4_step(ref_rhs, s, d), S0, T, 0.1)[-1]
    ref_gap = np.abs(ref - ref2)[:3]
    print(f"    reference check, RK4 dt=0.1 vs 0.05 differ by dh={ref_gap[0]:.1e} m, dV={ref_gap[1]:.1e} m/s")
    rows = []
    for dt in np.array([1.6, 0.8, 0.4, 0.2, 0.1, 0.05, 0.025]):
        s = team_euler(team_rhs, S0, dt, int(round(T / dt)))[-1]
        rows.append((dt, *np.abs(s - ref)[:3]))
    rows = np.array(rows)
    table(["dt [s]", "|dh| [m]", "|dV| [m/s]", "|dgamma| [rad]"], rows)
    ps = [tail_slope(rows[:, 0], rows[:, j]) for j in (1, 2, 3)]
    ok = all(0.85 < p < 1.15 for p in ps) and rows[-1, 2] < 1.0
    report("pipeline converges to the independent solution", ok,
           f"slopes h {ps[0]:.2f}, V {ps[1]:.2f}, gamma {ps[2]:.2f} (expect 1); |dV| = {rows[-1,2]:.2g} m/s at dt = {rows[-1,0]} s")
    return rows


def run_all():
    print("verification.py -- baseline: h0 = %.2f km, V0 = %.3f km/s, gamma0 = %.2f deg" %
          (H0 / 1e3, V0 / 1e3, np.rad2deg(GAMMA0)))
    check_euler()
    check_model()
    check_vacuum()
    check_allen_eggers()
    check_pipeline()
    n_fail = sum(not ok for _, ok in RESULTS)
    print(f"\n{len(RESULTS) - n_fail}/{len(RESULTS)} checks passed")
    return n_fail


if __name__ == "__main__":
    sys.exit(1 if run_all() else 0)

ModuleNotFoundError: No module named 'parameters'